# 05 — RWKV: Linear-Complexity Transformer

RWKV reformulates transformer computation to achieve linear complexity with transformer-level quality.

## Time-Mixing and Channel-Mixing

RWKV (Peng et al., 2023) reformulates the transformer to be computable as a recurrence with linear complexity, while maintaining transformer-style token mixing.

**Time-mixing** (replaces self-attention):
$$r_t = W_r \cdot (\mu_r x_t + (1-\mu_r) x_{t-1})$$
$$k_t = W_k \cdot (\mu_k x_t + (1-\mu_k) x_{t-1})$$
$$v_t = W_v \cdot (\mu_v x_t + (1-\mu_v) x_{t-1})$$
$$\text{wkv}_t = \frac{\exp(u + k_t) \cdot v_t + \sum_{s<t} \exp(w(t-1-s) + k_s) \cdot v_s}{\exp(u + k_t) + \sum_{s<t} \exp(w(t-1-s) + k_s)}$$
where *w* is a learned decay vector and *u* is a learned 'first token boost'.

The key insight: the denominator is a weighted sum that can be computed incrementally as a recurrence — O(1) per step at inference, O(n) total.

**Channel-mixing** (replaces MLP): mixes information across feature dimensions using a simple gated structure.

At inference, RWKV runs as a pure RNN — constant memory, constant compute per token. During training, the recurrence can be parallelised using a parallel prefix scan.

In [ ]:
# RWKV time-mixing and channel-mixing implementation
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class RWKVTimeMixing(nn.Module):
    """
    Simplified RWKV time-mixing block.
    """
    def __init__(self, d_model, layer_id=0):
        super().__init__()
        self.d_model = d_model

        # Learnable time-shift mixing ratios
        self.time_mix_k = nn.Parameter(torch.ones(d_model) * 0.5)
        self.time_mix_v = nn.Parameter(torch.ones(d_model) * 0.5)
        self.time_mix_r = nn.Parameter(torch.ones(d_model) * 0.5)

        # Decay and first-token boost
        self.time_decay = nn.Parameter(-5 + 8 * (torch.arange(d_model) / (d_model-1))**0.7)
        self.time_first = nn.Parameter(torch.zeros(d_model))

        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape

        # Time shift: mix current and previous token
        x_prev = torch.cat([torch.zeros(B, 1, D), x[:, :-1, :]], dim=1)
        xk = x * self.time_mix_k + x_prev * (1 - self.time_mix_k)
        xv = x * self.time_mix_v + x_prev * (1 - self.time_mix_v)
        xr = x * self.time_mix_r + x_prev * (1 - self.time_mix_r)

        k = self.key(xk)
        v = self.value(xv)
        r = torch.sigmoid(self.receptance(xr))

        # WKV computation (simplified sequential)
        w = -torch.exp(self.time_decay)  # (D,) — learned per-channel decay
        u = self.time_first  # (D,) — first token boost

        outputs = []
        a = torch.zeros(B, D)  # numerator accumulator
        b = torch.zeros(B, D)  # denominator accumulator

        for t in range(T):
            kt = k[:, t, :]  # (B, D)
            vt = v[:, t, :]

            # First-token boost for current position
            max_u_k = torch.maximum(u + kt, torch.zeros_like(kt))
            e1 = torch.exp(u + kt - max_u_k)
            e2 = torch.exp(w - max_u_k)  # simplified: a/b decay

            # WKV_t = (e^(u+k_t) * v_t + a) / (e^(u+k_t) + b)
            wkv = (e1 * vt + a) / (e1 + b + 1e-8)
            outputs.append(wkv)

            # Update accumulators
            a = e2 * a + torch.exp(kt - max_u_k) * vt
            b = e2 * b + torch.exp(kt - max_u_k)

        rwkv = r * torch.stack(outputs, dim=1)  # gated by receptance
        return self.output(rwkv)


class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(d_model) * 0.5)
        self.time_mix_r = nn.Parameter(torch.ones(d_model) * 0.5)
        self.key = nn.Linear(d_model, d_model * 4, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model * 4, d_model, bias=False)

    def forward(self, x):
        x_prev = torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], dim=1)
        xk = x * self.time_mix_k + x_prev * (1 - self.time_mix_k)
        xr = x * self.time_mix_r + x_prev * (1 - self.time_mix_r)
        k = torch.relu(self.key(xk))**2  # squared relu
        r = torch.sigmoid(self.receptance(xr))
        return r * self.value(k)


# Test RWKV blocks
torch.manual_seed(42)
tm = RWKVTimeMixing(d_model=32)
cm = RWKVChannelMixing(d_model=32)
x = torch.randn(2, 16, 32)
y_tm = tm(x)
y_cm = cm(x)
print(f'Time-mixing output: {y_tm.shape}')
print(f'Channel-mixing output: {y_cm.shape}')

total = sum(p.numel() for p in tm.parameters()) + sum(p.numel() for p in cm.parameters())
print(f'Total RWKV layer params: {total}')

In [ ]:
# RWKV inference: constant memory recurrent mode
import time
import matplotlib.pyplot as plt

# Demonstrate O(1) memory inference
class RWKVInference:
    """Stateful RWKV inference — constant memory per step."""
    def __init__(self, time_mixing):
        self.tm = time_mixing
        self.reset()

    def reset(self):
        d = self.tm.d_model
        self.a = torch.zeros(1, d)  # numerator state
        self.b = torch.zeros(1, d)  # denominator state
        self.x_prev = torch.zeros(1, d)

    def step(self, x_t):
        """Process one token, return output. O(1) memory."""
        xk = x_t * self.tm.time_mix_k + self.x_prev * (1 - self.tm.time_mix_k)
        xr = x_t * self.tm.time_mix_r + self.x_prev * (1 - self.tm.time_mix_r)
        k = self.tm.key(xk)
        v = self.tm.value(self.tm.value(xk) if hasattr(self.tm, 'value') else xk)  
        r = torch.sigmoid(self.tm.receptance(xr))
        w = -torch.exp(self.tm.time_decay)
        u = self.tm.time_first

        e1 = torch.exp(u + k)
        wkv = (e1 * self.a + self.b) / (e1 + torch.abs(self.a) + 1e-8)
        output = r * wkv

        self.a = torch.exp(w) * self.a + torch.exp(k) * self.a
        self.b = torch.exp(w) * self.b + torch.exp(k)
        self.x_prev = x_t
        return self.tm.output(output)

# Memory comparison: Transformer KV cache vs RWKV state
seq_lens = [100, 1000, 10000, 100000]
kv_cache_mb = [n * 2 * 32 * 4 * 4 / 1e6 for n in seq_lens]  # n * 2 * heads * d_head * float32
rwkv_state_mb = [32 * 4 / 1e6] * len(seq_lens)  # constant: just (a, b) vectors

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(seq_lens, kv_cache_mb, 'o-', label='Transformer KV cache O(n)', color='tomato')
ax.loglog(seq_lens, rwkv_state_mb, 's-', label='RWKV state O(1)', color='steelblue')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Memory (MB)')
ax.set_title('Inference Memory: Transformer vs RWKV')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/rwkv_memory.png', dpi=100, bbox_inches='tight')
plt.show()
print('RWKV memory comparison saved')